# PPE Compliance Checker — 01 Exploration

ITAI 1378 Midterm · initial experiments. **Run on Google Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU).

Goal of this notebook: get the environment working, pull a labeled PPE dataset, look at sample images and the class distribution, and run a pretrained YOLO11 model as a baseline before any fine-tuning.

## 1. Setup

In [ ]:
# Install core dependencies (Colab already has torch)
!pip install -q ultralytics roboflow

import ultralytics, torch
ultralytics.checks()
print('CUDA available:', torch.cuda.is_available())

## 2. Download a labeled PPE dataset

Pick one PPE dataset from Roboflow Universe (https://universe.roboflow.com/browse/construction/ppe), open it, and copy its `Download → YOLOv11` code snippet in place of the placeholder below. You'll need a free Roboflow API key.

Alternatively, use the built-in Ultralytics Construction-PPE dataset (no key needed) shown in the fallback cell.

In [ ]:
# OPTION A — Roboflow (paste your dataset's snippet; replace API key + slugs)
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('WORKSPACE').project('PPE-PROJECT')
# dataset = project.version(1).download('yolov11')
# DATA_YAML = f'{dataset.location}/data.yaml'

# OPTION B — fallback: Ultralytics Construction-PPE (auto-downloads, no key)
DATA_YAML = 'construction-ppe.yaml'
print('Using:', DATA_YAML)

## 3. Inspect classes and split sizes

In [ ]:
import yaml, os, glob

# For the Ultralytics fallback, the yaml ships with the package cache after first use.
# For Roboflow, DATA_YAML points to the downloaded folder.
try:
    with open(DATA_YAML) as f:
        cfg = yaml.safe_load(f)
    print('Classes:', cfg.get('names'))
    print('nc:', cfg.get('nc'))
except FileNotFoundError:
    print('data.yaml not found yet — run the download cell first (Option A or B).')

In [ ]:
# Count label instances per class across the training split (YOLO .txt labels)
import collections

def class_distribution(labels_dir, names):
    counts = collections.Counter()
    for txt in glob.glob(os.path.join(labels_dir, '*.txt')):
        with open(txt) as f:
            for line in f:
                if line.strip():
                    counts[int(line.split()[0])] += 1
    return {names[k] if isinstance(names, list) else names[k]: v for k, v in sorted(counts.items())}

# Adjust the path to your dataset's train/labels folder, then:
# print(class_distribution('PATH/train/labels', cfg['names']))

## 4. View a few sample images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import glob, random

# Point this at your dataset's train/images folder
img_paths = glob.glob('PATH/train/images/*.jpg')[:6]
if img_paths:
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    for ax, p in zip(axes.ravel(), random.sample(img_paths, min(6, len(img_paths)))):
        ax.imshow(Image.open(p)); ax.axis('off'); ax.set_title(os.path.basename(p), fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('Update the image path to your downloaded dataset.')

## 5. Pretrained YOLO11 baseline (sanity check)

Run the stock COCO-pretrained model on a sample image. COCO has a `person` class but **not** PPE items — this just confirms the pipeline works end-to-end before we fine-tune on PPE classes.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')          # nano = fastest; swap to yolo11s.pt for more accuracy
if img_paths:
    results = model(img_paths[0])
    results[0].show()               # shows detections (people, etc.)
else:
    print('Add a sample image path in cell 4 first.')

## 6. Next step (Week 2): fine-tune on PPE classes

This is the training command for the real model — kept here as a reference, not run in this exploration notebook.

In [ ]:
# Fine-tune YOLO11 on the PPE dataset (run in the training notebook)
# model = YOLO('yolo11s.pt')
# model.train(data=DATA_YAML, epochs=50, imgsz=640, batch=16)
# metrics = model.val()          # mAP@50, precision, recall per class
# print(metrics.box.map50)